# Data Source 02 · 부동산

| 출처 | 접근 | 검증 상태 |
|---|---|---|
| FHFA House Price Index CSV (`hpi_master`, `hpi_at_state` 등) | 무키 | A(파일명) / B(전체 URL 확인 필요) |
| 한국부동산원 R-ONE | 신청/무료키 | C |
| 국토교통부 실거래가 (공공데이터포털) | 무료키 | C |

FHFA 파일의 **정확한 URL은 FHFA HPI 다운로드 페이지에서 복사**해 `HPI_URL`에 넣는다(아래 값은 확인 필요한 추정 경로).
열 구성은 파일 버전마다 다를 수 있어 로더는 열 이름을 소문자로 정규화하고, 필수 열이 없으면 합성자료로 대체한다.

In [ ]:
import io, json, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt

def fetch_bytes(url, timeout=30):
    """무키 공개 URL 요청. 실패 시 예외 -> 호출부에서 합성 대체."""
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def synthetic_series(n, freq, seed, trend=0.05, season=12, noise=1.0, start="2015-01-01"):
    """NOT real data: 방법 시연용 합성 시계열."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    y = 100 + trend * t + 3 * np.sin(2 * np.pi * t / season) + rng.normal(0, noise, n)
    return pd.Series(y, index=pd.date_range(start, periods=n, freq=freq), name="value")

def quality_report(s):
    s = s.sort_index()
    print("관측 수:", len(s), "| 기간:", s.index.min(), "~", s.index.max())
    print("결측:", int(s.isna().sum()), "| 중복 시점:", int(s.index.duplicated().sum()))
    f = pd.infer_freq(s.index) if len(s) > 2 else None
    print("추정 주기:", f)

In [ ]:
def parse_fhfa_master(raw):
    df = pd.read_csv(io.BytesIO(raw) if isinstance(raw, (bytes, bytearray)) else io.StringIO(raw))
    df.columns = [c.strip().lower() for c in df.columns]
    need = {"yr", "period"}
    if not need.issubset(df.columns):
        raise ValueError("예상 열(yr, period) 없음: " + ", ".join(df.columns[:10]))
    return df

In [ ]:
HPI_URL = "https://www.fhfa.gov/hpi/download/monthly/hpi_master.csv"  # 확인 필요: 다운로드 페이지에서 실제 링크 복사
try:
    df = parse_fhfa_master(fetch_bytes(HPI_URL, timeout=60))
    print("열:", list(df.columns))
    DATA_KIND = "REAL: FHFA (열 선택은 아래에서 직접 확인)"
    # 열 이름 확인 후 직접 지정: 예) 지역/지수 구분 열로 필터링해 월별 시리즈 구성
except Exception as e:
    print("온라인 수집 실패 -> 합성 대체:", type(e).__name__, e)
    y = synthetic_series(240, "MS", 2026021, trend=0.4, season=12, noise=0.8, start="2005-01-01")
    DATA_KIND = "SYNTHETIC (NOT real data)"
    quality_report(y); y.plot(title=DATA_KIND); plt.show()
print("DATA_KIND:", DATA_KIND)

## 점검 체크리스트
- 지수의 기준시점·지수화 방식, 계절조정 여부 (SA/NSA)
- 지역 코드 체계(주/대도시권)와 패널 균형 여부
- 품질·면적 구성 변화(실거래 단위 자료일 때)
- 국내 자료(R-ONE, 실거래가)는 키·이용허락 확인 후 별도 셀로 추가